# Thyra: end-to-end validation workflow (zero-installation)

This notebook reproduces the complete Thyra workflow from the *Nature Methods* Correspondence
**“Thyra: Bridging Mass Spectrometry Imaging and SpatialData for Unified Multi-Modal Analysis”** -
converting a raw MSI acquisition into the SpatialData standard, quality-checking the result, and
spatially querying a region of interest: entirely in the browser.

**How to run:** `Runtime ▸ Run all`. No installation or configuration is required on your machine.

- Source code (MIT): https://github.com/M4i-Imaging-Mass-Spectrometry/thyra
- Documentation: https://M4i-Imaging-Mass-Spectrometry.github.io/thyra
- Example dataset: https://doi.org/10.5281/zenodo.18326569

## 1. Install Thyra and the scverse tools

In [ ]:
# Download ONLY the MSI archive (19.1 GB) from the public Zenodo record.
# NOTE: this is large; on Colab expect a long download. A small subset file,
# if added to the record, is the better target for a quick validation run.
!mkdir -p data
!wget -c -O data/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz "https://zenodo.org/records/18326569/files/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz?download=1"
!tar -xzf data/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz -C data/
!find data -name "*.imzML" -o -name "*.ibd" | head

### Or bring your own data

The workflow is not tied to the example dataset. Any imzML acquisition (the `.imzML` and `.ibd` pair together) can go through the same steps. Two ways to get yours into Colab:

1. **Small files:** run the upload cell below and pick both files.
2. **Large files:** put them in your Google Drive and mount it, which avoids slow browser uploads.

Then set `imzml` to your file's path in Section 3 and run the rest unchanged. If conversion reports "Pixel size not found in metadata", pass --pixel-size (CLI) or the pixel size argument (API). Bruker `.d` and Waters `.raw` folders work too if you zip and upload them the same way.

In [ ]:
# Option 1: upload your own .imzML + .ibd pair (small files)
# from google.colab import files
# up = files.upload()          # select BOTH the .imzML and the .ibd file
# imzml = [f for f in up if f.endswith('.imzML')][0]

# Option 2: mount Google Drive (large files)
# from google.colab import drive
# drive.mount('/content/drive')
# imzml = '/content/drive/MyDrive/path/to/your_data.imzML'

## 2. Download the example dataset (public, Zenodo)

imzML acquisitions come as a pair: the `.imzML` metadata file **and** the `.ibd` binary file.

In [ ]:
# Download ONLY the MSI archive (19.1 GB) from the public Zenodo record.
# NOTE: this is large; on Colab expect a long download. A small subset file,
# if added to the record, is the better target for a quick validation run.
!mkdir -p data
!wget -c -O data/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz "https://zenodo.org/records/18326569/files/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz?download=1"
!tar -xzf data/MALDI-MSI_Sagittal_Mouse_Brain.tar.gz -C data/
!find data -name "*.imzML" -o -name "*.ibd" | head

## 3. Convert MSI → SpatialData

Thyra auto-detects the input format, extracts the full spectra and instrument metadata, harmonizes the mass axis with a physics-aware resampling strategy, and streams the data into SpatialData elements (ion images, spectral matrix, pixel coordinates).

In [ ]:
import glob
imzml = glob.glob("data/*.imzML")[0]   # TODO: pin the exact filename as deposited
print("Converting:", imzml)

from thyra import convert_msi
ok = convert_msi(imzml, "output/thyra_example.zarr")
print("Conversion succeeded:", ok)

## 4. Load and inspect the SpatialData object

In [ ]:
import spatialdata as sd
sdata = sd.read_zarr("output/thyra_example.zarr")
print(sdata)
print(sdata.coordinate_systems)

## 5. Quality control

Because the output is a standard SpatialData/OME-NGFF object, QC is transparent and scriptable.

In [ ]:
import numpy as np
adata = sdata.tables[list(sdata.tables.keys())[0]]

tic = np.asarray(adata.X.sum(axis=1)).ravel()          # total ion current per pixel
print("TIC  median=%.3g  CV=%.2f%%" % (np.median(tic), 100*tic.std()/tic.mean()))
low = tic < np.percentile(tic, 1)
print("Low-signal pixels:", int(low.sum()))

In [ ]:
# Visualize an ion image inline
import matplotlib.pyplot as plt
img_key = list(sdata.images.keys())[0]
img = sdata.images[img_key]
try:
    arr = img.compute().data if hasattr(img, "compute") else np.asarray(img)
    plt.figure(figsize=(6,5)); plt.imshow(np.squeeze(arr)[0] if arr.ndim==3 else np.squeeze(arr), cmap="viridis")
    plt.title(f"Ion image: {img_key}"); plt.axis("off"); plt.colorbar(); plt.show()
except Exception as e:
    print("Adjust to your element names: see sdata overview above.", e)

## 6. ROI query: co-localized molecular profile

Define a region of interest in the shared coordinate system and pull its mean spectrum (its metabolic fingerprint). With aligned modalities in the same object, the identical query returns cell-type composition or morphology.

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon
import spatialdata as sd, numpy as np

# TODO: set ROI coordinates appropriate to the example tissue (µm, shared coordinate system)
x0, y0, x1, y1 = 100, 100, 600, 600
roi = gpd.GeoDataFrame({"region": ["ROI_1"]},
                       geometry=[Polygon([(x0,y0),(x1,y0),(x1,y1),(x0,y1)])])

sub = sd.polygon_query(sdata, roi.geometry[0], target_coordinate_system=list(sdata.coordinate_systems)[0])
adata = sub.tables[list(sub.tables.keys())[0]]
mean_spectrum = np.asarray(adata.X.mean(axis=0)).ravel()
top = np.argsort(mean_spectrum)[-10:][::-1]
print("Top m/z features in ROI_1:", adata.var_names[top].tolist())

## 7. Where to go next

- **Interactive, graphical inspection:** the converted `.zarr` store opens directly in **napari**
  (desktop) and **Vitessce** (browser): see the documentation for one-line loaders.
- **Downstream analysis:** the spectral table is a standard AnnData object, so Scanpy/Squidpy
  workflows apply directly (normalization, clustering, spatial statistics).
- **CLI without any environment management:** `uvx thyra input.imzML output.zarr` (or `pipx run thyra ...`).

*Questions or issues:* https://github.com/M4i-Imaging-Mass-Spectrometry/thyra/issues